# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sudais2211/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

I will rank content pages for CTR optimization using a simple baseline action score. The score prioritizes pages with meaningful search exposure and an observed CTR opportunity relative to their average search position.

Pages with sufficient impressions and below-expected CTR for their position are prioritized for CTR review. Pages with high exposure but insufficient evidence of a CTR opportunity receive a lower priority.

This is a directional decision-support rule, not a causal prediction.

Reason codes
HIGH_EXPOSURE_CTR_OPPORTUNITY — high search exposure with a measured CTR opportunity.
MEDIUM_EXPOSURE_CTR_OPPORTUNITY — moderate search exposure with a measured CTR opportunity.
LOW_EXPOSURE_CTR_OPPORTUNITY — lower exposure but an observed CTR opportunity.
NO_CLEAR_OPPORTUNITY — insufficient evidence of a strong CTR opportunity.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ML-07 — Load the FlyRank dataset efficiently
# ============================================================

from google.colab import userdata
from datasets import load_dataset
import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# 1. Get Hugging Face token from Colab Secrets
# ------------------------------------------------------------

try:
    HF_TOKEN = userdata.get("HF-TOKEN")
    print("Hugging Face token loaded.")
except Exception:
    HF_TOKEN = None
    print("HF_TOKEN not found. Trying public dataset access.")

# ------------------------------------------------------------
# 2. Load the dataset
# ------------------------------------------------------------

print("Loading FlyRank dataset...")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    name="fact_content_daily_performance",
    split="train",
    token=HF_TOKEN
)

print("Dataset loaded successfully.")
print("Number of rows:", ds.num_rows)
print("Number of columns:", ds.num_columns)

print("\nColumns:")
print(ds.column_names)

Hugging Face token loaded.
Loading FlyRank dataset...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset loaded successfully.
Number of rows: 78835655
Number of columns: 30

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
print("Dataset type:", type(ds))

print("\nNumber of rows:", ds.num_rows)
print("Number of columns:", ds.num_columns)

print("\nALL COLUMNS:")
for i, col in enumerate(ds.column_names):
    print(i, col)

print("\nFIRST ROW:")
print(ds[0])

Dataset type: <class 'datasets.arrow_dataset.Dataset'>

Number of rows: 78835655
Number of columns: 30

ALL COLUMNS:
0 report_date
1 client_hash_id
2 content_hash_id
3 client_has_gsc
4 client_has_ga4
5 gsc_data_available
6 ga4_data_available
7 gsc_impressions
8 gsc_clicks
9 gsc_sum_position
10 gsc_avg_position
11 ga4_pageviews
12 ga4_sessions
13 ga4_users
14 ga4_engaged_sessions
15 ga4_total_engagement_sec
16 sessions_organic
17 sessions_direct
18 sessions_referral
19 sessions_social
20 sessions_paid
21 sessions_ai
22 ai_chatgpt
23 ai_perplexity
24 ai_gemini
25 ai_copilot
26 ai_claude
27 ai_meta
28 ai_other
29 scroll_events

FIRST ROW:
{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pagev

## 3. Top-20 review

*I reviewed the highest-ranked 20 pages from the baseline queue. The action is based on observed exposure, average position, and the measured CTR gap relative to the position benchmark.

The confidence note reflects the strength of the observed evidence, not causal certainty.

A recommendation could be wrong if the page has unusual search intent, a small or unstable observation history, SERP features that affect clicks, branded versus non-branded query differences, or other factors not represented in the available data.*

In [4]:
print("Variables currently available:")
print([x for x in globals().keys() if not x.startswith("_")])

print("\nDoes queue exist?", "queue" in globals())

import os
print("\nDoes baseline CSV exist?",
      os.path.exists("work/outputs/baseline_action_score.csv"))

Variables currently available:
['In', 'Out', 'get_ipython', 'exit', 'quit', 'pd', 'np', 'START_DATE', 'END_DATE', 'USE_COLUMNS', 'CHUNK_SIZE', 'march_parts', 'userdata', 'load_dataset', 'os', 'HF_TOKEN', 'ds', 'i', 'col']

Does queue exist? False

Does baseline CSV exist? False


In [6]:
# ============================================================
# ML-07 — Memory-safe March 2026 extraction
# ============================================================

import pandas as pd
import numpy as np
import gc

START_DATE = "2026-03-01"
END_DATE = "2026-04-01"

USE_COLUMNS = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

CHUNK_SIZE = 100_000

march_parts = []

print("Starting batch processing...")
print("Dataset rows:", ds.num_rows)

for start in range(0, ds.num_rows, CHUNK_SIZE):

    end = min(
        start + CHUNK_SIZE,
        ds.num_rows
    )

    # Get small batch
    batch = ds.select(
        range(start, end)
    )

    # Select only required columns
    batch_df = batch.select_columns(
        USE_COLUMNS
    ).to_pandas()

    # IMPORTANT:
    # report_date is datetime.date,
    # so compare with date strings
    batch_df = batch_df[
        (batch_df["report_date"].astype(str) >= START_DATE) &
        (batch_df["report_date"].astype(str) < END_DATE)
    ]

    if len(batch_df) > 0:
        march_parts.append(batch_df)

    # Progress every 1 million rows
    if end % 1_000_000 < CHUNK_SIZE:
        print(
            f"Processed {end:,} / {ds.num_rows:,} rows"
        )

    # Delete temporary batch
    del batch
    del batch_df
    gc.collect()

# Combine March data
if march_parts:
    march_df = pd.concat(
        march_parts,
        ignore_index=True
    )
else:
    march_df = pd.DataFrame(
        columns=USE_COLUMNS
    )

print("\n================================")
print("March extraction complete")
print("================================")

print("March rows:", len(march_df))
print("Shape:", march_df.shape)

if len(march_df) > 0:

    print(
        "Date range:",
        march_df["report_date"].min(),
        "to",
        march_df["report_date"].max()
    )

    print("\nColumns:")
    print(march_df.columns.tolist())

    display(march_df.head())

Starting batch processing...
Dataset rows: 78835655
Processed 1,000,000 / 78,835,655 rows
Processed 2,000,000 / 78,835,655 rows
Processed 3,000,000 / 78,835,655 rows
Processed 4,000,000 / 78,835,655 rows
Processed 5,000,000 / 78,835,655 rows
Processed 6,000,000 / 78,835,655 rows
Processed 7,000,000 / 78,835,655 rows
Processed 8,000,000 / 78,835,655 rows
Processed 9,000,000 / 78,835,655 rows
Processed 10,000,000 / 78,835,655 rows
Processed 11,000,000 / 78,835,655 rows
Processed 12,000,000 / 78,835,655 rows
Processed 13,000,000 / 78,835,655 rows
Processed 14,000,000 / 78,835,655 rows
Processed 15,000,000 / 78,835,655 rows
Processed 16,000,000 / 78,835,655 rows
Processed 17,000,000 / 78,835,655 rows
Processed 18,000,000 / 78,835,655 rows
Processed 19,000,000 / 78,835,655 rows
Processed 20,000,000 / 78,835,655 rows
Processed 21,000,000 / 78,835,655 rows
Processed 22,000,000 / 78,835,655 rows
Processed 23,000,000 / 78,835,655 rows
Processed 24,000,000 / 78,835,655 rows
Processed 25,000,000 

,report_date,client_hash_id,content_hash_id,gsc_data_available,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,20,0,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,1,0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,125,1,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,7,0,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,11,0,2.272727


In [7]:
# ============================================================
# ML-07 — Prepare March data for CTR analysis
# ============================================================

df = march_df.copy()

# Keep only rows where GSC data is available
df = df[
    (df["gsc_data_available"] == True) &
    (df["gsc_impressions"] > 0) &
    (df["gsc_avg_position"].notna()) &
    (df["gsc_avg_position"] > 0)
].copy()

# Calculate observed CTR
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"]
)

# Remove invalid CTR values
df = df[
    (df["ctr"] >= 0) &
    (df["ctr"] <= 1)
].copy()

print("Usable rows:", len(df))

display(
    df[
        [
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position"
        ]
    ].head()
)

Usable rows: 3447872


,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
0,content_b7e512995f79d5a6,20,0,0.000000,3.350000
2,content_7a105f548d9c6916,125,1,0.008000,4.928000
3,content_905aa32a0230694e,7,0,0.000000,4.000000
4,content_a3ea9792f793ec72,11,0,0.000000,2.272727
5,content_36c36abc7650d7af,239,1,0.004184,7.347280


In [8]:
# ============================================================
# SIGNAL CHECK 1 — CTR vs Average Position
# ============================================================

# Create position buckets
df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=[
        "TOP_3",
        "POSITION_4_10",
        "POSITION_11_20",
        "POSITION_21_PLUS"
    ],
    include_lowest=True
)

# Calculate statistics
signal_1 = (
    df.groupby(
        "position_bucket",
        observed=True
    )
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        median_position=("gsc_avg_position", "median")
    )
    .reset_index()
)

print("SIGNAL CHECK 1 — CTR vs Average Position")
print("=" * 60)

display(signal_1)

SIGNAL CHECK 1 — CTR vs Average Position


,position_bucket,n,median_ctr,mean_ctr,median_position
0,TOP_3,564173,0.0,0.004918,1.935484
1,POSITION_4_10,1456122,0.0,0.003473,5.868601
2,POSITION_11_20,519223,0.0,0.002770,14.000000
3,POSITION_21_PLUS,908354,0.0,0.001289,37.086957


In [9]:
# ============================================================
# SIGNAL CHECK 2 — Search Exposure / Impressions
# ============================================================

# Create four exposure groups
df["exposure_bucket"] = pd.qcut(
    df["gsc_impressions"],
    q=4,
    labels=[
        "LOW_EXPOSURE",
        "MEDIUM_LOW_EXPOSURE",
        "MEDIUM_HIGH_EXPOSURE",
        "HIGH_EXPOSURE"
    ],
    duplicates="drop"
)

# Calculate statistics
signal_2 = (
    df.groupby(
        "exposure_bucket",
        observed=True
    )
    .agg(
        n=("ctr", "size"),
        median_impressions=("gsc_impressions", "median"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print("SIGNAL CHECK 2 — Search Exposure")
print("=" * 60)

display(signal_2)

SIGNAL CHECK 2 — Search Exposure


,exposure_bucket,n,median_impressions,median_ctr,mean_ctr
0,LOW_EXPOSURE,953936,2.0,0.0,0.003766
1,MEDIUM_LOW_EXPOSURE,805975,10.0,0.0,0.002374
2,MEDIUM_HIGH_EXPOSURE,827519,34.0,0.0,0.002781
3,HIGH_EXPOSURE,860442,156.0,0.0,0.003060


In [10]:
# ============================================================
# ML-07 — BASELINE ACTION SCORE
# ============================================================

# Calculate median CTR benchmark for each position bucket
position_benchmark = (
    df.groupby(
        "position_bucket",
        observed=True
    )["ctr"]
    .median()
)

print("Position CTR Benchmarks:")
display(position_benchmark.to_frame("benchmark_ctr"))

# Assign benchmark to each row
df["expected_ctr"] = df["position_bucket"].map(
    position_benchmark
)

# Calculate CTR gap
df["ctr_gap"] = (
    df["expected_ctr"] - df["ctr"]
)

# Keep only positive opportunity
df["ctr_gap_positive"] = df["ctr_gap"].clip(
    lower=0
)

# Calculate relative CTR gap
df["ctr_gap_pct"] = np.where(
    df["expected_ctr"] > 0,
    df["ctr_gap_positive"] / df["expected_ctr"],
    0
)

# Prevent extreme values
df["ctr_gap_pct"] = df["ctr_gap_pct"].clip(
    lower=0,
    upper=1
)

# Log-transform impressions
df["exposure_score"] = np.log1p(
    df["gsc_impressions"]
)

# Normalize exposure
max_exposure = df["exposure_score"].max()

if max_exposure > 0:
    df["exposure_score_normalized"] = (
        df["exposure_score"] /
        max_exposure
    )
else:
    df["exposure_score_normalized"] = 0

# Final baseline score
# 70% CTR opportunity
# 30% search exposure

df["action_score"] = (
    0.70 * df["ctr_gap_pct"] +
    0.30 * df["exposure_score_normalized"]
)

print("Baseline score calculated.")

display(
    df[
        [
            "content_hash_id",
            "gsc_impressions",
            "ctr",
            "gsc_avg_position",
            "expected_ctr",
            "ctr_gap_pct",
            "action_score"
        ]
    ]
    .sort_values(
        "action_score",
        ascending=False
    )
    .head(10)
)

Position CTR Benchmarks:


,benchmark_ctr
position_bucket,
TOP_3,0.0
POSITION_4_10,0.0
POSITION_11_20,0.0
POSITION_21_PLUS,0.0


Baseline score calculated.


,content_hash_id,gsc_impressions,ctr,gsc_avg_position,expected_ctr,ctr_gap_pct,action_score
9655081,content_44f34c0a90047651,40084,0.000025,0.083350,0.0,0.0,0.300000
8054586,content_eadb33b5df496f4a,39305,0.006411,2.197507,0.0,0.0,0.299445
103612,content_34a70fea29d15f24,39003,0.000051,2.764916,0.0,0.0,0.299226
9179913,content_eadb33b5df496f4a,38436,0.007051,2.195988,0.0,0.0,0.298812
103661,content_945d6ff91386c817,37368,0.000000,8.613948,0.0,0.0,0.298014
8972299,content_eadb33b5df496f4a,35404,0.006355,2.188397,0.0,0.0,0.296486
8640840,content_eadb33b5df496f4a,34817,0.006405,2.181348,0.0,0.0,0.296013
9767993,content_eadb33b5df496f4a,34606,0.006791,2.242501,0.0,0.0,0.295841
7406798,content_eadb33b5df496f4a,33571,0.006404,2.309046,0.0,0.0,0.294981
8882581,content_fec55986a1868d62,33383,0.000000,0.181500,0.0,0.0,0.294822


In [11]:
# ============================================================
# REASON CODES AND ACTION LABELS
# ============================================================

# Use median impressions as the exposure threshold
exposure_threshold = df["gsc_impressions"].median()

df["reason_code"] = np.select(
    [
        (df["ctr_gap_pct"] >= 0.30) &
        (df["gsc_impressions"] >= exposure_threshold),

        df["ctr_gap_pct"] >= 0.20,

        df["ctr_gap_pct"] > 0
    ],
    [
        "HIGH_CTR_GAP_HIGH_EXPOSURE",
        "MEANINGFUL_CTR_GAP",
        "SMALL_CTR_GAP"
    ],
    default="NO_CLEAR_OPPORTUNITY"
)

# Action labels
df["action"] = np.select(
    [
        df["reason_code"] ==
        "HIGH_CTR_GAP_HIGH_EXPOSURE",

        df["reason_code"] ==
        "MEANINGFUL_CTR_GAP",

        df["reason_code"] ==
        "SMALL_CTR_GAP"
    ],
    [
        "PRIORITIZE_CTR_REVIEW",
        "REVIEW_CTR",
        "MONITOR_CTR"
    ],
    default="NO_ACTION"
)

print("Reason codes and actions created.")

print("\nAction distribution:")
print(df["action"].value_counts())

print("\nReason code distribution:")
print(df["reason_code"].value_counts())

Reason codes and actions created.

Action distribution:
action
NO_ACTION    3447872
Name: count, dtype: int64

Reason code distribution:
reason_code
NO_CLEAR_OPPORTUNITY    3447872
Name: count, dtype: int64


In [12]:
# ============================================================
# CREATE RANKED QUEUE
# ============================================================

# Sort by baseline score
queue = df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

# Add rank
queue["rank"] = np.arange(
    1,
    len(queue) + 1
)

# Select final columns
queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "expected_ctr",
        "ctr_gap",
        "ctr_gap_pct",
        "action_score",
        "reason_code",
        "action"
    ]
].copy()

print("Ranked queue created.")
print("Total rows:", len(queue))

display(queue.head(20))

Ranked queue created.
Total rows: 3447872


,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,expected_ctr,ctr_gap,ctr_gap_pct,action_score,reason_code,action
0,1,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-28,40084,1,0.000025,0.083350,0.0,-0.000025,0.0,0.300000,NO_CLEAR_OPPORTUNITY,NO_ACTION
1,2,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-29,39305,252,0.006411,2.197507,0.0,-0.006411,0.0,0.299445,NO_CLEAR_OPPORTUNITY,NO_ACTION
2,3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,2026-03-04,39003,2,0.000051,2.764916,0.0,-0.000051,0.0,0.299226,NO_CLEAR_OPPORTUNITY,NO_ACTION
3,4,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-28,38436,271,0.007051,2.195988,0.0,-0.007051,0.0,0.298812,NO_CLEAR_OPPORTUNITY,NO_ACTION
4,5,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368,0,0.000000,8.613948,0.0,0.000000,0.0,0.298014,NO_CLEAR_OPPORTUNITY,NO_ACTION
5,6,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-30,35404,225,0.006355,2.188397,0.0,-0.006355,0.0,0.296486,NO_CLEAR_OPPORTUNITY,NO_ACTION
6,7,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-27,34817,223,0.006405,2.181348,0.0,-0.006405,0.0,0.296013,NO_CLEAR_OPPORTUNITY,NO_ACTION
7,8,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-31,34606,235,0.006791,2.242501,0.0,-0.006791,0.0,0.295841,NO_CLEAR_OPPORTUNITY,NO_ACTION
8,9,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-24,33571,215,0.006404,2.309046,0.0,-0.006404,0.0,0.294981,NO_CLEAR_OPPORTUNITY,NO_ACTION
9,10,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-30,33383,0,0.000000,0.181500,0.0,0.000000,0.0,0.294822,NO_CLEAR_OPPORTUNITY,NO_ACTION


In [13]:
# ============================================================
# SAVE BASELINE ACTION SCORE
# ============================================================

import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)

output_path = (
    "work/outputs/baseline_action_score.csv"
)

queue.to_csv(
    output_path,
    index=False
)

print("SUCCESS!")
print("=" * 60)
print("Saved:", output_path)
print("Rows:", len(queue))
print("Columns:", len(queue.columns))

display(queue.head(20))

SUCCESS!
Saved: work/outputs/baseline_action_score.csv
Rows: 3447872
Columns: 14


,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,expected_ctr,ctr_gap,ctr_gap_pct,action_score,reason_code,action
0,1,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-28,40084,1,0.000025,0.083350,0.0,-0.000025,0.0,0.300000,NO_CLEAR_OPPORTUNITY,NO_ACTION
1,2,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-29,39305,252,0.006411,2.197507,0.0,-0.006411,0.0,0.299445,NO_CLEAR_OPPORTUNITY,NO_ACTION
2,3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,2026-03-04,39003,2,0.000051,2.764916,0.0,-0.000051,0.0,0.299226,NO_CLEAR_OPPORTUNITY,NO_ACTION
3,4,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-28,38436,271,0.007051,2.195988,0.0,-0.007051,0.0,0.298812,NO_CLEAR_OPPORTUNITY,NO_ACTION
4,5,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368,0,0.000000,8.613948,0.0,0.000000,0.0,0.298014,NO_CLEAR_OPPORTUNITY,NO_ACTION
5,6,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-30,35404,225,0.006355,2.188397,0.0,-0.006355,0.0,0.296486,NO_CLEAR_OPPORTUNITY,NO_ACTION
6,7,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-27,34817,223,0.006405,2.181348,0.0,-0.006405,0.0,0.296013,NO_CLEAR_OPPORTUNITY,NO_ACTION
7,8,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-31,34606,235,0.006791,2.242501,0.0,-0.006791,0.0,0.295841,NO_CLEAR_OPPORTUNITY,NO_ACTION
8,9,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-24,33571,215,0.006404,2.309046,0.0,-0.006404,0.0,0.294981,NO_CLEAR_OPPORTUNITY,NO_ACTION
9,10,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-30,33383,0,0.000000,0.181500,0.0,0.000000,0.0,0.294822,NO_CLEAR_OPPORTUNITY,NO_ACTION


In [14]:
# ============================================================
# TOP-20 REVIEW
# ============================================================

top20 = queue.head(20).copy()

top20["confidence_note"] = np.select(
    [
        top20["ctr_gap_pct"] >= 0.30,
        top20["ctr_gap_pct"] >= 0.20
    ],
    [
        "Higher confidence: large observed CTR gap.",
        "Medium confidence: meaningful observed CTR gap."
    ],
    default="Lower confidence: weaker observed CTR evidence."
)

top20["what_would_make_it_wrong"] = (
    "Could be wrong if search intent, SERP features, "
    "brand effects, or limited observation history explain "
    "the observed CTR difference."
)

display(top20)

,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,expected_ctr,ctr_gap,ctr_gap_pct,action_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-28,40084,1,0.000025,0.083350,0.0,-0.000025,0.0,0.300000,NO_CLEAR_OPPORTUNITY,NO_ACTION,Lower confidence: weaker observed CTR evidence.,"Could be wrong if search intent, SERP features..."
1,2,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-29,39305,252,0.006411,2.197507,0.0,-0.006411,0.0,0.299445,NO_CLEAR_OPPORTUNITY,NO_ACTION,Lower confidence: weaker observed CTR evidence.,"Could be wrong if search intent, SERP features..."
2,3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,2026-03-04,39003,2,0.000051,2.764916,0.0,-0.000051,0.0,0.299226,NO_CLEAR_OPPORTUNITY,NO_ACTION,Lower confidence: weaker observed CTR evidence.,"Could be wrong if search intent, SERP features..."
3,4,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-28,38436,271,0.007051,2.195988,0.0,-0.007051,0.0,0.298812,NO_CLEAR_OPPORTUNITY,NO_ACTION,Lower confidence: weaker observed CTR evidence.,"Could be wrong if search intent, SERP features..."
4,5,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368,0,0.000000,8.613948,0.0,0.000000,0.0,0.298014,NO_CLEAR_OPPORTUNITY,NO_ACTION,Lower confidence: weaker observed CTR evidence.,"Could be wrong if search intent, SERP features..."
5,6,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-30,35404,225,0.006355,2.188397,0.0,-0.006355,0.0,0.296486,NO_CLEAR_OPPORTUNITY,NO_ACTION,Lower confidence: weaker observed CTR evidence.,"Could be wrong if search intent, SERP features..."
6,7,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-27,34817,223,0.006405,2.181348,0.0,-0.006405,0.0,0.296013,NO_CLEAR_OPPORTUNITY,NO_ACTION,Lower confidence: weaker observed CTR evidence.,"Could be wrong if search intent, SERP features..."
7,8,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-31,34606,235,0.006791,2.242501,0.0,-0.006791,0.0,0.295841,NO_CLEAR_OPPORTUNITY,NO_ACTION,Lower confidence: weaker observed CTR evidence.,"Could be wrong if search intent, SERP features..."
8,9,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-24,33571,215,0.006404,2.309046,0.0,-0.006404,0.0,0.294981,NO_CLEAR_OPPORTUNITY,NO_ACTION,Lower confidence: weaker observed CTR evidence.,"Could be wrong if search intent, SERP features..."
9,10,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-30,33383,0,0.000000,0.181500,0.0,0.000000,0.0,0.294822,NO_CLEAR_OPPORTUNITY,NO_ACTION,Lower confidence: weaker observed CTR evidence.,"Could be wrong if search intent, SERP features..."


## 4. Weak picks + leakage check

The weakest picks are pages that rank highly despite having limited exposure or weak evidence of a meaningful CTR opportunity. These cases should be reviewed carefully because the baseline is intentionally simple.

The baseline does not use future-window outcomes. It uses March 2026 observations only. It also does not use FlyRank product flags or manually generated action labels as model inputs. The action label is generated directly from the baseline score and reason code.

In [15]:
# ============================================================
# 4. WEAK PICKS + LEAKAGE CHECK
# ============================================================

print("WEAK PICKS")
print("=" * 70)

# Identify high-ranked pages with weak evidence
weak_picks = queue[
    (
        (queue["gsc_impressions"] < queue["gsc_impressions"].median()) &
        (queue["ctr_gap_pct"] < 0.20)
    )
].copy()

# Show the first 10 weak candidates
weak_picks = weak_picks.head(10)

print("Number of weak candidates identified:", len(weak_picks))

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "gsc_impressions",
            "ctr",
            "gsc_avg_position",
            "expected_ctr",
            "ctr_gap_pct",
            "action_score",
            "reason_code",
            "action"
        ]
    ]
)

WEAK PICKS
Number of weak candidates identified: 10


,rank,content_hash_id,gsc_impressions,ctr,gsc_avg_position,expected_ctr,ctr_gap_pct,action_score,reason_code,action
1726056,1726057,content_858ed2d4874548ac,17,0.000000,3.470588,0.0,0.0,0.081813,NO_CLEAR_OPPORTUNITY,NO_ACTION
1726057,1726058,content_e732bf260e17575e,17,0.000000,10.823529,0.0,0.0,0.081813,NO_CLEAR_OPPORTUNITY,NO_ACTION
1726058,1726059,content_b5491e512f7ce8e9,17,0.000000,14.176471,0.0,0.0,0.081813,NO_CLEAR_OPPORTUNITY,NO_ACTION
1726059,1726060,content_6fe13b6f0ecc6607,17,0.000000,9.647059,0.0,0.0,0.081813,NO_CLEAR_OPPORTUNITY,NO_ACTION
1726060,1726061,content_88f3cdd163b68c37,17,0.000000,13.705882,0.0,0.0,0.081813,NO_CLEAR_OPPORTUNITY,NO_ACTION
1726061,1726062,content_4afdf85d19181779,17,0.000000,6.823529,0.0,0.0,0.081813,NO_CLEAR_OPPORTUNITY,NO_ACTION
1726062,1726063,content_1b963da98d291805,17,0.000000,12.764706,0.0,0.0,0.081813,NO_CLEAR_OPPORTUNITY,NO_ACTION
1726063,1726064,content_66f765855bbe8bfb,17,0.000000,23.588235,0.0,0.0,0.081813,NO_CLEAR_OPPORTUNITY,NO_ACTION
1726064,1726065,content_0fd5951442096856,17,0.000000,66.647059,0.0,0.0,0.081813,NO_CLEAR_OPPORTUNITY,NO_ACTION
1726065,1726066,content_cfdde5449b8664c1,17,0.235294,5.176471,0.0,0.0,0.081813,NO_CLEAR_OPPORTUNITY,NO_ACTION


In [16]:
# ============================================================
# LEAKAGE CHECK
# ============================================================

print("\nLEAKAGE CHECK")
print("=" * 70)

# Columns used to calculate the baseline score
score_inputs = [
    "gsc_impressions",
    "ctr",
    "gsc_avg_position",
    "expected_ctr",
    "ctr_gap_pct"
]

# Confirm score inputs exist
print("Score inputs used:")
for col in score_inputs:
    print(" -", col)

# Confirm no future-window columns are used
future_keywords = [
    "future",
    "next",
    "label",
    "outcome",
    "trend"
]

possible_leakage = [
    col for col in df.columns
    if any(
        keyword in col.lower()
        for keyword in future_keywords
    )
]

print("\nPotential future/label-related columns present in df:")
print(possible_leakage)

# Check whether FlyRank product flags are being used
product_flag_columns = [
    col for col in df.columns
    if "flag" in col.lower()
]

print("\nProduct flag columns present:")
print(product_flag_columns)

print("\nConclusion:")
print(
    "The baseline score was calculated from March 2026 observed "
    "GSC impressions, clicks/CTR, and average position. "
    "No future-window outcome was used as a score input."
)


LEAKAGE CHECK
Score inputs used:
 - gsc_impressions
 - ctr
 - gsc_avg_position
 - expected_ctr
 - ctr_gap_pct

Potential future/label-related columns present in df:
[]

Product flag columns present:
[]

Conclusion:
The baseline score was calculated from March 2026 observed GSC impressions, clicks/CTR, and average position. No future-window outcome was used as a score input.


## Self-check

March 2026 observations
        ↓
CTR calculation
        ↓
Position benchmark
        ↓
CTR gap
        ↓
Action score
        ↓
Reason code
        ↓
Action label